# BEH 00: setup

Settings and helpers for the rolling customer behaviour dataset. Every BEH notebook runs this first. Change a default here, or set the variable in the calling notebook before its %run.


In [ ]:
%%pyspark
# Behavioural dataset settings. Every BEH notebook runs this notebook first (%run BEH_00_Setup).
# A value the calling notebook or pipeline has already set wins; anything not set takes the default here.
# Decisions behind the defaults: BehaviouralSegmentation\out\clarifications.md and out\design.md.
target_schema       = globals().get("target_schema")       or "beh"         # output tables
stg_schema          = globals().get("stg_schema")          or "stg"         # working tables (stg.beh_*)
ods_schema          = globals().get("ods_schema")          or "ods_v2"
ads_schema          = globals().get("ads_schema")          or "ads_v2"
link_schema         = globals().get("link_schema")         or "ods"         # ods.scveadmlinkfinal (policy -> SCV)
hist_start          = globals().get("hist_start")          or "2021-07-31"  # first month-end built (history for 24m windows)
first_obs           = globals().get("first_obs")           or "2023-07-31"  # first analysis month-end (IsAnalysisMonth = 1)
last_me             = globals().get("last_me")             or "2026-08-31"  # last complete month-end in the sources
renewal_outcome_end = globals().get("renewal_outcome_end") or "2026-08-31"  # renewals due up to here have settled flags
outcome_end         = globals().get("outcome_end")         or "2026-08-31"  # forward outcome windows must end by here
cov_events          = globals().get("cov_events")          or "2021-07-01"  # ads_v2 event stream start
cov_quotes          = globals().get("cov_quotes")          or "2026-05-06"  # quotes reach policyholder SCV keys from here (GDPR wipe before)
cov_calls           = globals().get("cov_calls")           or "2022-11-01"  # Genesys calls (and call-derived events) at full volume
cov_mychill         = globals().get("cov_mychill")         or "2024-03-01"  # MyChill document workflow events start (Feb 2024)
cov_arrears         = globals().get("cov_arrears")         or "2023-01-01"  # arrears events built in volume
cov_mta_count       = globals().get("cov_mta_count")       or "2021-04-01"  # PolicyMtaDay MTA counts (Relay postings, ods_v2 cut-off 2021-04-01)
cov_mta             = globals().get("cov_mta")             or "2026-01-22"  # PolicyMta change detail (history starts 2026-01-22)
cov_loan            = globals().get("cov_loan")            or "2021-07-01"  # PolicyLoanStatus, read from the ads start
cov_comms           = globals().get("cov_comms")           or "2022-12-01"  # campaign history at full volume
cov_consent         = globals().get("cov_consent")         or "2021-01-01"  # Mercury consent, contact-centre apps logged consistently from 2021
cov_portal          = globals().get("cov_portal")          or "2021-04-01"  # renewal portal log-ins (RenewalPortalEvent from Apr 2021)
cov_docwf           = globals().get("cov_docwf")           or "2024-01-01"  # MyChill document workflow decisions at volume (DocumentWorkflowStatus)
cov_mfq             = globals().get("cov_mfq")             or "2021-07-01"  # MFQ quotes linked to policyholders (ods_v2.QuoteMotorQuery from 2021-04)
cov_mfq_device      = globals().get("cov_mfq_device")      or "2024-01-01"  # MFQ first / last device per quote (QuoteMotorSummary from 2024)
cov_web             = globals().get("cov_web")             or "2023-04-01"  # warehouse GA4 clicks (ods_v2.WebSession) start
cov_tcall           = globals().get("cov_tcall")           or "2022-11-01"  # TelephonyCall queues under today's names at volume
cov_mfq_purchase    = globals().get("cov_mfq_purchase")    or "2024-01-01"  # MFQ quotes linked to the policy bought (ods_v2.Quote.ConvertedToPolicy)
cov_friction_v3     = globals().get("cov_friction_v3")     or "2022-11-01"  # friction extract v3.3 episodes from Jul 2022; call-wait measures from Nov 2022
friction_v3_table   = globals().get("friction_v3_table")   or "ods.friction_episode_v3_20260902"  # friction extract v3.3, the copy for this data date
batch_id            = globals().get("batch_id")            or ""


In [ ]:
%%pyspark
import datetime
batch_id = batch_id or datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
conf = {"beh.target": target_schema, "beh.stg": stg_schema, "beh.ods": ods_schema, "beh.ads": ads_schema,
        "beh.link": link_schema, "beh.hist_start": hist_start, "beh.first_obs": first_obs, "beh.last_me": last_me,
        "beh.renewal_outcome_end": renewal_outcome_end, "beh.outcome_end": outcome_end,
        "beh.cov_events": cov_events, "beh.cov_quotes": cov_quotes, "beh.cov_calls": cov_calls,
        "beh.cov_mta": cov_mta, "beh.cov_loan": cov_loan, "beh.cov_comms": cov_comms, "beh.cov_mychill": cov_mychill,
        "beh.cov_arrears": cov_arrears, "beh.cov_mta_count": cov_mta_count, "beh.cov_consent": cov_consent,
        "beh.cov_portal": cov_portal, "beh.cov_docwf": cov_docwf, "beh.cov_mfq": cov_mfq,
        "beh.cov_mfq_device": cov_mfq_device, "beh.cov_web": cov_web, "beh.cov_tcall": cov_tcall,
        "beh.cov_mfq_purchase": cov_mfq_purchase, "beh.cov_friction_v3": cov_friction_v3,
        "beh.friction_v3": friction_v3_table, "beh.batch_id": batch_id}
spark.conf.set("spark.sql.variable.substitute", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
for k, v in conf.items():
    spark.conf.set(k, v)
spark.sql(f"create schema if not exists {target_schema}")
print(conf)


In [ ]:
%%pyspark
from pyspark.sql import functions as F


def beh_run_log(notebook, status, detail=""):
    """One row per notebook start / finish in <target>.BehRunLog."""
    spark.sql(f"create table if not exists {target_schema}.BehRunLog "
              "(BatchId string, Notebook string, Status string, Detail string, LoggedAt timestamp) using delta")
    spark.createDataFrame([(batch_id, notebook, status, detail)], "BatchId string, Notebook string, Status string, Detail string") \
         .withColumn("LoggedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.BehRunLog")


def beh_check(notebook, spec):
    """Row count, duplicate keys and null keys per table; every result goes to <target>.BehBuildLog, then the cell
    fails if any table failed, with the likely cause."""
    rows, failures = [], []
    for s in spec:
        t = s["table"].format(stg=stg_schema, target=target_schema)
        df = spark.table(t)
        key = s["key"]
        agg = df.agg(F.count(F.lit(1)).alias("n"),
                     F.countDistinct(F.struct(*key)).alias("d"),
                     F.sum(F.when(F.expr(" or ".join(f"`{c}` is null" for c in key)), 1).otherwise(0)).alias("nulls")).first()
        problems = []
        if agg["n"] == 0:
            problems.append("empty: its source is empty or a filter / join removed every row (check the dates in BEH_00_Setup)")
        if agg["d"] != agg["n"]:
            problems.append(f"{agg['n'] - agg['d']:,} duplicate keys on {' + '.join(key)}: a join fanned out or a source "
                            f"sent a record twice. Find them: select {', '.join(key)}, count(*) from {t} group by "
                            f"{', '.join(key)} having count(*) > 1")
        if agg["nulls"]:
            problems.append(f"{agg['nulls']:,} rows with a null key column ({', '.join(key)})")
        rows.append((batch_id, notebook, t, int(agg["n"]), "ok" if not problems else "FAILED", "; ".join(problems)))
        if problems:
            failures.append(f"{t}: " + "; ".join(problems))
        print(f"{t}: {agg['n']:,} rows, {'ok' if not problems else 'FAILED'}")
    spark.sql(f"create table if not exists {target_schema}.BehBuildLog (BatchId string, Notebook string, TableName string, "
              "RowCount bigint, Result string, Detail string, LoggedAt timestamp) using delta")
    if rows:
        spark.createDataFrame(rows, "BatchId string, Notebook string, TableName string, RowCount bigint, Result string, Detail string") \
             .withColumn("LoggedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.BehBuildLog")
    if failures:
        beh_run_log(notebook, "failed", " | ".join(failures)[:4000])
        raise Exception("Checks failed:\n" + "\n".join(failures))
